In [1]:
import os
import pandas as pd
import varseek as vk

In [6]:
w = 37
k = 41

data_dir = os.path.join(os.path.dirname(os.getcwd()), "data_varseek")
vk_ref_dir = os.path.join(data_dir, "vk_ref_out")
reference_dir = os.path.join(data_dir, "reference")

tcga_dir = os.path.join(reference_dir, "tcga")
tcga_maf_path = os.path.join(tcga_dir, "mc3.v0.2.8.PUBLIC.maf")
tcga_csv_path = os.path.join(tcga_dir, "tcga_mc3.csv")

cosmic_dir = os.path.join(reference_dir, "cosmic")

dbsnp_dir = os.path.join(reference_dir, "dbsnp")

In [4]:
for out_dir in [data_dir, reference_dir, tcga_dir, cosmic_dir, dbsnp_dir, vk_ref_dir]:
    os.makedirs(out_dir, exist_ok=True)

## TCGA

In [ ]:
# TCGA
tcga_url = "https://api.gdc.cancer.gov/data/1c8cfe5f-e52d-41ba-94da-f15ea1337efc"

if not os.path.exists(tcga_maf_path):
    !wget -O {tcga_maf_path}.gz {tcga_url}
    !gunzip {tcga_maf_path}.gz

if not os.path.exists(tcga_csv_path):
    tcga_df = pd.read_csv(tcga_maf_path, sep="\t", comment="#", low_memory=False, nrows=1000)  #!!! remove nrows to read full file
    tcga_df[["Transcript_ID", "HGVSc", "cDNA_position", "ENSP", "HGVSp", "Gene", "Hugo_Symbol", "Chromosome", "Start_Position", "End_Position", "Strand", "Reference_Allele", "Tumor_Seq_Allele1", "Tumor_Seq_Allele2", "dbSNP_RS", "all_effects"]].to_csv(tcga_csv_path, index=False)

In [ ]:
mutation_pattern = r"(?:c|g)\.([0-9_\-\+\*\(\)\?]+)([a-zA-Z>]+)"

def cds_to_cdna(tcga_df, var_column="HGVSc"):    
    tcga_df = tcga_df.copy()

    if "variant_type" not in tcga_df.columns:
        vk.utils.add_variant_type(tcga_df, var_column=var_column)  # Add variant_type column based on var_column

    tcga_df[["nucleotide_positions", "actual_variant"]] = tcga_df[var_column].str.extract(mutation_pattern)  # Extract nucleotide positions and mutation info from Mutation CDS
    tcga_df = tcga_df.dropna(subset=["nucleotide_positions", "actual_variant"])  # Filter out tcga_df that did not match the re
    split_positions = tcga_df["nucleotide_positions"].str.split("_", expand=True)  # Split nucleotide positions into start and end positions

    tcga_df["start_variant_position"] = split_positions[0]
    if split_positions.shape[1] > 1:
        tcga_df["end_variant_position"] = split_positions[1].fillna(split_positions[0])
    else:
        tcga_df["end_variant_position"] = tcga_df["start_variant_position"]

    tcga_df.loc[tcga_df["end_variant_position"].isna(), "end_variant_position"] = tcga_df["start_variant_position"]

    tcga_df[["start_variant_position", "end_variant_position"]] = tcga_df[["start_variant_position", "end_variant_position"]].astype(int)

    tcga_df["start_variant_position_cdna"] = tcga_df["cDNA_position"]
    tcga_df["end_variant_position_cdna"] = tcga_df["end_variant_position"] + (tcga_df["start_variant_position_cdna"] - tcga_df["start_variant_position"])

    tcga_df["variant_length"] = tcga_df["end_variant_position"] - tcga_df["start_variant_position"] + 1

    tcga_df[["start_variant_position_cdna", "end_variant_position_cdna"]] = tcga_df[["start_variant_position_cdna", "end_variant_position_cdna"]].astype(str)

    conditions = [
        tcga_df["variant_type"] == "substitution",  # Substitution
        (tcga_df["variant_type"] == "deletion") & (tcga_df["variant_length"] == 1),  # Single base deletion
        (tcga_df["variant_type"] == "deletion") & (tcga_df["variant_length"] > 1),  # Multi base deletion
        tcga_df["variant_type"] == "insertion",  # Insertion
        tcga_df["variant_type"] == "delins",  # Delins
        (tcga_df["variant_type"] == "duplication") & (tcga_df["variant_length"] == 1),  # Single base duplication
        (tcga_df["variant_type"] == "duplication") & (tcga_df["variant_length"] > 1),  # Multi base duplication
        tcga_df["variant_type"] == "inversion"  # Inversion
    ]

    # Define corresponding variant formats
    choices = [
        "c." + tcga_df["start_variant_position_cdna"] + tcga_df["actual_variant"] + ">" + tcga_df["ALT"],  # Substitution
        "c." + tcga_df["start_POS_deletion"] + "del",  # Single base deletion
        "c." + tcga_df["start_POS_deletion"] + "_" + tcga_df["end_POS_for_multibase_deletion_and_delins_and_inversion"] + "del",  # Multi base deletion
        "c." + tcga_df["POS"] + "_" + tcga_df["end_POS_for_insertion"] + "ins" + tcga_df["ALT_first_base_trimmed"],  # Insertion
        "c." + tcga_df["POS"] + "_" + tcga_df["end_POS_for_multibase_deletion_and_delins_and_inversion"] + "delins" + tcga_df["ALT"],  # Delins
        "c." + tcga_df["POS"] + "dup",  # Single base duplication
        "c." + tcga_df["POS"] + "_" + tcga_df["end_POS_for_multibase_deletion_and_delins_and_inversion"] + "inv"  # Inversion
    ]

    # Apply np.select
    tcga_df[var_column] = np.select(conditions, choices, default="g.unknown")  # Default to None if no match
    tcga_df.drop(columns=["REF_len", "ALT_len", "ALT_RC", "start_POS_deletion", "start_POS_deletion_starting_at_1", "end_POS_for_multibase_deletion_and_delins_and_inversion", "end_POS_for_multibase_deletion_starting_at_1", "end_POS_for_insertion", "ALT_first_base_trimmed", "ALT_last_base_trimmed"], inplace=True, errors="ignore")

    tcga_df["POS"] = tcga_df["POS"].astype('Int64')

## COSMIC CMC

## dbSNP

In [ ]:
dbsnp_vcf_url = "https://ftp.ncbi.nih.gov/snp/latest_release/VCF/GCF_000001405.40.gz"  # .40 = GRCh38; .25 = GRCh37
dbsnp_tbi_url = "https://ftp.ncbi.nih.gov/snp/latest_release/VCF/GCF_000001405.40.gz.tbi"